# Operationalisation of GAL

Thesis: *Corpus Construction*, section *Operationalisation of GAL*.

**Requirements**
- Environment: `easy-language` (`environment-easy-language.yml`)
- Packages: spaCy 3.8 with the model `de_core_news_md`, pyphen
- Data: `data/corpus/sd_prompts.csv`, `data/corpus/ls_prompts.csv` via `corpus_spec`, `data/corpus/ls_drafts.csv` (in this repository)
- GPU and network: not needed

**Writes:** `exports/ls_metrics_{sd,drafts,ls}.csv`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "corpus").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src" / "corpus"))

import datetime
import statistics

import corpus_spec as spec
from readability import syllable_count
import pandas as pd
from IPython.display import display

## Gate D: DIN SPEC criteria and readability formulas

Every GAL frame must meet seven countable criteria derived from DIN SPEC 33429, such as one finite verb per sentence and no subordinate clause; readability formulas are reported alongside. 140 of 144 GAL frames pass; the four failures of `reinigungskraft_muell_leeren` are a known tagger artefact. The German4All drafts pass in 17 of 36 cases.

In [2]:
OUT_TEMPLATE = "exports/ls_metrics_{}.csv"
SET_LABELS = {"sd": "GSL", "drafts": "German4All drafts", "ls": "GAL"}
DRAFTS_PATH = Path("data/corpus/ls_drafts.csv")
NLP_MODEL = "de_core_news_md"          # spaCy 3.8 German pipeline (morphologizer for Case/VerbForm)
MAX_WORDS_PER_SENTENCE = 10

CRITERIA = ["multi_statement", "subclause", "passive", "genitive", "pers_pronoun_3",
            "comma", "long_sentence"]
FINITE_TAGS = {"VVFIN", "VAFIN", "VMFIN"}  # STTS finite-verb tags — redundant check alongside
                                            # morph VerbForm=Fin, see finite-detection note below
FIELDS = (["item_id", "set", "profession", "scene_label", "text", "n_sentences", "n_words"]
          + CRITERIA + ["gate_pass", "gate_fails", "wsf1", "flesch_de", "lix",
                        "nlp_model", "run_date"])

_NLP = None


def nlp():
    global _NLP
    if _NLP is None:
        import spacy
        _NLP = spacy.load(NLP_MODEL)
    return _NLP

In [3]:
def criteria_profile(text: str) -> dict:
    doc = nlp()(text.replace("Der*die ", "Die ").replace("*", ""))
    counts = dict.fromkeys(CRITERIA, 0)
    n_sentences = 0
    for sent in doc.sents:
        n_sentences += 1
        finite = sum(1 for t in sent
                     if "Fin" in t.morph.get("VerbForm") or t.tag_ in FINITE_TAGS)
        if finite != 1:
            counts["multi_statement"] += 1
        if len(sent.text.split()) > MAX_WORDS_PER_SENTENCE:
            counts["long_sentence"] += 1
        has_werden_aux = any(t.pos_ == "AUX" and t.lemma_ == "werden" for t in sent)
        has_participle = any(t.tag_ == "VVPP" for t in sent)
        if has_werden_aux and has_participle:
            counts["passive"] += 1
        for t in sent:
            if t.pos_ == "SCONJ" or "Rel" in t.morph.get("PronType"):
                counts["subclause"] += 1
            if "Gen" in t.morph.get("Case"):
                counts["genitive"] += 1
            if (t.tag_ == "PPER" and "3" in t.morph.get("Person")) or t.tag_ == "PPOSAT":
                counts["pers_pronoun_3"] += 1
            if t.text in {",", ";"}:
                counts["comma"] += 1
    counts["n_sentences"] = n_sentences
    return counts


def word_stats(texts: list[str]) -> dict:
    """Pooled token statistics for the formulas (words = whitespace tokens, punctuation stripped per
    word for length/syllables — consistent with spec's n_words convention).
    """
    n_sentences = sum(criteria_cache[t]["n_sentences"] for t in texts)
    words = [w for t in texts for w in t.split()]
    clean = ["".join(c for c in w if c.isalpha()) for w in words]
    clean = [w for w in clean if w]
    syls = [syllable_count(w) for w in clean]
    return {
        "n_sentences": n_sentences,
        "n_words": len(clean),
        "sl": len(clean) / n_sentences,                                # mean sentence length
        "asw": sum(syls) / len(clean),                                 # syllables per word
        "ms": 100 * sum(1 for s in syls if s >= 3) / len(clean),       # % >=3-syllable words
        "iw": 100 * sum(1 for w in clean if len(w) > 6) / len(clean),  # % >6-letter words
        "es": 100 * sum(1 for s in syls if s == 1) / len(clean),       # % 1-syllable words
    }


def formulas(texts: list[str]) -> dict:
    s = word_stats(texts)
    return {
        "wsf1": round(0.1935 * s["ms"] + 0.1672 * s["sl"] + 0.1297 * s["iw"]
                      - 0.0327 * s["es"] - 0.875, 2),
        "flesch_de": round(180 - s["sl"] - 58.5 * s["asw"], 2),
        "lix": round(s["sl"] + s["iw"], 2),
    }


criteria_cache: dict[str, dict] = {}


def analyze(items: list[dict], set_name: str) -> list[dict]:
    run_date = datetime.date.today().isoformat()
    rows = []
    for it in items:
        prof_counts = criteria_profile(it["text"])
        criteria_cache[it["text"]] = prof_counts
        fails = [c for c in CRITERIA if prof_counts[c] > 0]
        rows.append({
            "item_id": it["item_id"], "set": set_name, "profession": it["profession"],
            "scene_label": it["scene_label"], "text": it["text"],
            "n_sentences": prof_counts["n_sentences"], "n_words": len(it["text"].split()),
            **{c: prof_counts[c] for c in CRITERIA},
            "gate_pass": not fails, "gate_fails": ";".join(fails),
            **formulas([it["text"]]),          # per-frame formula values = screening only
            "nlp_model": NLP_MODEL, "run_date": run_date,
        })
    return rows


def sd_items() -> list[dict]:
    return [{"item_id": f"{r['profession']}_{r['scene_label']}_{r['condition']}",
             "profession": r["profession"], "scene_label": r["scene_label"],
             "text": r["prompt_text"]}
            for r in spec.unique_test_prompts()]


def draft_items() -> list[dict]:
    drafts = pd.read_csv(DRAFTS_PATH, dtype=str, keep_default_na=False)
    return [{"item_id": f"{r['item_id']}_d{r['draft_idx']}", "profession": r["profession"],
             "scene_label": r["scene_label"], "text": r["draft_text"]}
            for r in drafts.to_dict("records")]


def ls_items() -> list[dict] | None:
    frames = getattr(spec, "LS_FRAMES", None)
    if frames is None:
        return None
    return [{"item_id": f"{prof}_{label}_{form}", "profession": prof,
             "scene_label": label, "text": spec.ls_sentence(template, prof, form)}
            for prof, scenes in frames.items()
            for label, template in scenes.items()
            for form in spec.GENDER_FORMS]


def report(rows: list[dict], set_name: str) -> None:
    texts = [r["text"] for r in rows]
    passed = [r for r in rows if r["gate_pass"]]
    print(f"\n=== {SET_LABELS[set_name]} ({len(rows)} frames) ===")
    print(f"criteria gate: {len(passed)}/{len(rows)} pass"
          + ("" if set_name != "sd" else "  (GSL baseline, not expected to pass)"))
    viol = {c: sum(r[c] for r in rows) for c in CRITERIA}
    print("violations:", ", ".join(f"{c}={n}" for c, n in viol.items() if n) or "none")
    pooled = formulas(texts)
    print(f"pooled formulas: WSF1={pooled['wsf1']}  Flesch-DE={pooled['flesch_de']}  "
          f"LIX={pooled['lix']}   (conventions: <=7 / >=80 / <35)")
    per_profession = {prof: formulas([r["text"] for r in rows if r["profession"] == prof])
                      for prof in spec.PROFESSIONS if any(r["profession"] == prof for r in rows)}
    display(pd.DataFrame(per_profession).T)


def paired_report(sd_rows: list[dict], ls_rows: list[dict]) -> None:
    print("\n=== GAL against GSL per scene and gender form (formula deltas, GAL minus GSL) ===")
    sd_by_id = {r["item_id"]: r for r in sd_rows}
    deltas = []
    for r in ls_rows:
        sd = sd_by_id.get(r["item_id"])
        if sd:
            deltas.append({m: r[m] - sd[m] for m in ("wsf1", "flesch_de", "lix")})
    if not deltas:
        print("no matching scenes")
        return
    table = []
    for m, better in (("wsf1", "lower"), ("flesch_de", "higher"), ("lix", "lower")):
        vals = [d[m] for d in deltas]
        right_dir = sum(1 for v in vals if (v < 0) == (better == "lower"))
        table.append({"formula": m, "mean delta": round(statistics.mean(vals), 2),
                      "pairs simpler in GAL": f"{right_dir}/{len(vals)}", "simpler is": better})
    display(pd.DataFrame(table).set_index("formula"))

In [4]:
SET = None  # "sd", "drafts" or "ls"; None runs every available set

In [5]:
sets = {}
if SET in (None, "sd"):
    sets["sd"] = sd_items()
if SET in (None, "drafts") and DRAFTS_PATH.exists():
    sets["drafts"] = draft_items()
if SET in (None, "ls"):
    items = ls_items()
    if items:
        sets["ls"] = items
    elif SET == "ls":
        raise RuntimeError("spec.LS_FRAMES not defined yet.")

all_rows = {}
for set_name, items in sets.items():
    rows = analyze(items, set_name)
    all_rows[set_name] = rows
    out = Path(OUT_TEMPLATE.format(set_name))
    pd.DataFrame(rows, columns=FIELDS).to_csv(out, index=False, lineterminator="\r\n")
    report(rows, set_name)
    print(f"-> {out}")

if "sd" in all_rows and "ls" in all_rows:
    paired_report(all_rows["sd"], all_rows["ls"])


=== GSL (144 frames) ===
criteria gate: 61/144 pass  (GSL baseline, not expected to pass)
violations: multi_statement=1, genitive=108, long_sentence=62
pooled formulas: WSF1=9.86  Flesch-DE=48.11  LIX=47.42   (conventions: <=7 / >=80 / <35)


,wsf1,flesch_de,lix
mechaniker,10.50,32.00,46.23
elektriker,10.01,55.33,48.93
reinigungskraft,10.66,47.26,50.28
florist,10.79,48.72,54.95
baecker,7.85,52.80,41.91
fotograf,9.63,48.07,41.63


-> exports/ls_metrics_sd.csv

=== German4All drafts (36 frames) ===
criteria gate: 17/36 pass
violations: multi_statement=15, subclause=14, genitive=2, pers_pronoun_3=10, comma=18, long_sentence=4
pooled formulas: WSF1=2.79  Flesch-DE=83.95  LIX=24.63   (conventions: <=7 / >=80 / <35)


,wsf1,flesch_de,lix
mechaniker,4.46,67.15,29.67
elektriker,3.43,89.62,29.17
fotograf,3.01,87.38,22.31


-> exports/ls_metrics_drafts.csv



=== GAL (144 frames) ===
criteria gate: 140/144 pass
violations: multi_statement=4
pooled formulas: WSF1=8.76  Flesch-DE=54.85  LIX=42.49   (conventions: <=7 / >=80 / <35)


,wsf1,flesch_de,lix
mechaniker,11.46,32.88,47.81
elektriker,9.39,57.72,45.85
reinigungskraft,8.45,58.52,43.07
florist,8.28,64.06,46.90
baecker,6.09,61.06,32.39
fotograf,9.05,54.44,39.66


-> exports/ls_metrics_ls.csv

=== GAL against GSL per scene and gender form (formula deltas, GAL minus GSL) ===


,mean delta,pairs simpler in GAL,simpler is
formula,,,
wsf1,-1.10,89/144,lower
flesch_de,7.14,108/144,higher
lix,-4.62,104/144,lower
